# G8 對抗資料產生：真標題 + Gemini 合成反極（依 fit_type 分流）

> 內文**全取自外部新聞**（爬蟲產出的 `adversarial_base`，避免與訓練集內文重複）；只用 Gemini 合成標題。

## 配對策略：真標題保留、合成補缺的那極

爬來的真實新聞大多 label=0，依人工標的 `fit_type` 分流，補上模型缺的那極：

| fit_type | 真標題（原樣保留） | Gemini 合成 |
|----------|------------------|------------|
| `transition` | 真 label=0 → `tw_neg`（轉折詞反例，label=0） | 同內文造**驚訝類轉折 clickbait** → `tw_pos`（label=1） |
| `academic`   | 真 label=0 保留 | 同內文造**知性腔 clickbait**（label=1） |
| `normal`     | 真標題保留（依原 label） | 同內文**平實改寫**成反極 |
| `none`       | 真標題保留 | **不套特殊句式**，僅通用改寫（硬新聞硬套會製造假懸念、標錯 label） |

**轉折詞均勻覆蓋**：transition 的 tw_pos prompt 明確要求驚訝類轉折詞（竟/竟然/居然/沒想到/不料…）
輪流均勻使用，避免集中在單一詞造成分布偏斜。

## Checkpoint（斷點續跑）
每筆 Gemini 結果**逐筆 flush 進 jsonl**，啟動時讀回 `done_ids` 跳過已完成。
中斷後重跑只補未完成的，不重燒已花的 API 額度。**要全部重跑請手動刪 checkpoint 檔。**

## Step 1：載入爬蟲產出的 adversarial_base，過濾 keep=1、按 fit_type 分組

In [6]:
import json
import os
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from google import genai

# --- 輸入：爬蟲產出的 adversarial_base（真實新聞內文）---
# 跑量後改成真實檔案 glob；現用 fixture 驗證 pipeline 邏輯
RAW_DIR = Path("../dataset/raw_scraped")
INPUT_GLOB = "*adversarial_base.csv"   # 跑量後會 match 各站點檔案

frames = []
for p in sorted(RAW_DIR.glob(INPUT_GLOB)):
    frames.append(pd.read_csv(p, dtype=str))
    print(f"載入 {p.name}: {len(frames[-1])} 列")
if not frames:
    raise FileNotFoundError(f"找不到 {RAW_DIR}/{INPUT_GLOB}；請先跑爬蟲 --use adversarial_base")

base = pd.concat(frames, ignore_index=True)

# 只保留 keep=1（廢棄樣本標 keep=0 不刪，見 ANNOTATION_GUIDE）
base = base[base["keep"].astype(str) == "1"].reset_index(drop=True)
# label 容錯轉 int
base["label"] = base["label"].fillna("0").astype(int)
# fit_type 空值視為 none
base["fit_type"] = base["fit_type"].fillna("none").replace("", "none")
# gemini 欄位：0=只保留真標題不走合成，1=走完整 Gemini 流程（預設 1，向後相容舊檔案）
base["gemini"] = base["gemini"].fillna("1").astype(int) if "gemini" in base.columns else 1

print("\nfit_type 分布：")
print(base["fit_type"].value_counts())
print("\nlabel 分布：")
print(base["label"].value_counts())
print("\ngemini 分布：")
print(base["gemini"].value_counts())

載入 adversarial_base.csv: 358 列

fit_type 分布：
fit_type
none          146
normal         84
transition     64
academic       61
clickbait       3
Name: count, dtype: int64

label 分布：
label
0    276
1     82
Name: count, dtype: int64

gemini 分布：
gemini
1    316
0     42
Name: count, dtype: int64


## Step 2：Gemini client 與四種 fit_type 的 prompt

In [7]:
load_dotenv(Path("../backend/.env"))
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

G8_MODEL_ID   = "gemini-3.1-flash-lite"
G8_BATCH_SIZE = 5
G8_SLEEP      = 8
G8_RETRIES    = 3
G8_RETRY_SLEEP = 15
G8_CHECKPOINT = Path("../dataset/processed/_adv_g8_checkpoint.jsonl")

# 驚訝類轉折詞：transition+label=0 由程式輪流指定，保證 8 詞均勻覆蓋（根治 G7「但93%/竟0%」偏斜）。
SURPRISE_WORDS_ZH = ["竟", "竟然", "居然", "沒想到", "不料", "萬萬沒想到", "驚見", "赫然"]
SURPRISE_WORDS_EN = ["you won't believe", "shockingly", "turns out", "unexpectedly",
                     "to everyone's surprise", "what happened next", "little did they know"]


def _content_snippet(r, n=300):
    c = r.get("content", "")
    return (str(c)[:n]) if isinstance(c, str) or c is not None else ""


# ═══════════════════════════════════════════════════════════════════════
# 合成品質驗證器（Codex review 2026-06：Human Gate 前先用程式擋掉壞樣本）
#   問題：build_prompt_clickbait 太保守 → Gemini 常產「正常摘要」而非 clickbait，
#         污染 G8（把 non-clickbait 標成 synthetic label=1）。
#   對策：cb 合成標題必須命中「資訊落差訊號」；tw_pair neg/pos 都必須帶指定轉折詞。
#         驗證失敗 → 主迴圈重試一次，再不行丟棄（synth=None / 該對作廢）。
# ═══════════════════════════════════════════════════════════════════════

# 資訊落差訊號：clickbait 標題至少命中一個才算「有遮蔽」。
#   中文：模糊代稱/懸念詞/疑問詞；英文用 regex 詞界比對，避免子字串誤判
#   （如 "this" 命中 "thistle"）。英文以「單字級 vague 詞 + wh 詞 + 懸念片語」三組。
import re

GAP_SIGNALS_ZH = [
    "這", "那", "某", "一個", "幾個", "原因", "真相", "內幕", "結果", "答案",
    "為何", "為什麼", "竟", "居然", "沒想到", "背後", "曝光", "揭", "全看",
    "做了", "說了", "發生", "這一幕", "如何", "怎麼", "什麼", "誰", "？", "?",
]
# 英文：單字級 vague / wh 詞（用 \b 詞界）
GAP_WORDS_EN = [
    "this", "these", "that", "those", "one", "some", "something",
    "someone", "what", "why", "how", "where", "when", "who",
    "decision", "reason", "moment", "message", "outcome", "truth", "secret",
    "revealed", "exploring", "quietly", "behind",
]
# 英文：懸念片語（直接子字串）
GAP_PHRASES_EN = [
    "the reason", "the way", "what happened", "the science behind",
    "a decision", "a certain", "one reason",
    "caught off guard", "you won't believe", "?",
]


def _norm(s):
    return (s or "").strip().lower()


def _en_has_gap(s):
    low = _norm(s)
    if any(p in low for p in GAP_PHRASES_EN):
        return True
    toks = set(re.findall(r"[a-z']+", low))
    return any(w in toks for w in GAP_WORDS_EN)


def _count_gap_signals(s, lang):
    """計算標題命中的 gap signal 數量，用於比較 synth 是否比 orig 更模糊。"""
    if lang == "zh":
        return sum(1 for n in GAP_SIGNALS_ZH if n in s)
    else:
        low = _norm(s)
        toks = set(re.findall(r"[a-z']+", low))
        phrase_hits = sum(1 for p in GAP_PHRASES_EN if p in low)
        word_hits = sum(1 for w in GAP_WORDS_EN if w in toks)
        return phrase_hits + word_hits


def cb_has_gap(synth, orig_title, lang):
    """cb/academic 合成標題的資訊落差檢查。回傳 (ok: bool, reason: str)。
    不合格條件：
      (a) 空 / 太短；
      (b) 完全沒命中任何 gap signal（= 只是平實摘要）；
      (c) 與原標題過度相似（直接抄，沒有遮蔽動作）；
      (d) orig 本身已有 gap signal，但 synth 的 gap signal 數量不超過 orig
          （= 反向訓練：聳動原標題 label=0，生出比原標題更平淡的 synth label=1）。
    """
    if not synth or not str(synth).strip():
        return False, "empty"
    s = str(synth).strip()
    if len(s) < (6 if lang == "zh" else 12):
        return False, "too_short"
    has = any(n in s for n in GAP_SIGNALS_ZH) if lang == "zh" else _en_has_gap(s)
    if not has:
        return False, "no_gap_signal"
    # 與原標題過度相似 → 視為沒做遮蔽
    a, b = set(str(orig_title or "")), set(s)
    if a and b:
        jac = len(a & b) / len(a | b)
        if jac > 0.85:
            return False, f"too_similar({jac:.2f})"
    # 反向訓練防護：orig 本身就很聳動時，synth 必須比 orig 有更多 gap signal
    # 避免「聳動 label=0 原標題 vs 平淡 label=1 synth」的反向對比
    if orig_title:
        orig_gaps = _count_gap_signals(str(orig_title), lang)
        synth_gaps = _count_gap_signals(s, lang)
        if orig_gaps >= 2 and synth_gaps <= orig_gaps:
            return False, f"synth_not_gappier(orig={orig_gaps},synth={synth_gaps})"
    return True, "ok"


def pair_ok(neg, pos, word, lang):
    """tw_pair 檢查：neg/pos 都要在、都要含指定轉折詞，且 pos 要有資訊落差。
    回傳 (ok, reason)。Codex #3：原本只在 prompt 要求、程式沒驗，neg 常漏轉折詞。"""
    if not neg or not str(neg).strip() or not pos or not str(pos).strip():
        return False, "missing_side"
    neg_s, pos_s = str(neg).strip(), str(pos).strip()
    w = word if lang == "zh" else _norm(word)
    if w not in (neg_s if lang == "zh" else _norm(neg_s)):
        return False, "neg_missing_hook"
    if w not in (pos_s if lang == "zh" else _norm(pos_s)):
        return False, "pos_missing_hook"
    gap_ok, _ = cb_has_gap(pos_s, "", lang)   # pos 必須是 clickbait（有落差）
    if not gap_ok:
        return False, "pos_no_gap"
    return True, "ok"


# ── 處理矩陣（依 Bug 修復需求 + PLAN §2）──
#   label=0 → 造反極 label=1（依 fit_type 句式）
#   label=1 → 改平實 label=0（tone_plain，沿用 G7 改寫準則以控制 prompt 變因）
#   transition+label=0 特殊：一個 prompt 造「一對」tw_neg(0,轉折詞+忠實) + tw_pos(1,轉折詞+懸念)，
#     同內文同轉折詞，才能教模型「轉折詞≠clickbait，差別在資訊落差」（真正修 Bug 3）。

def build_prompt_plain(batch_rows, lang):
    """label=1 → 中性平實版（tone_plain, label=0）。沿用 G7 tone_plain 改寫準則。"""
    numbered = "\n".join(
        f"{i+1}. 標題：{r['title']}\n   內文：{_content_snippet(r)}" for i, r in enumerate(batch_rows)
    )
    if lang == "zh":
        prompt = f"""以下有 {len(batch_rows)} 個標題黨標題（含內文）。請將每個改寫成中性、客觀、
可作為非標題黨新聞標題的一句繁體中文。
要求：忠實反映內文、不得加入內文沒有的資訊；去除誇飾詞、懸念、問句、感嘆、模糊代稱。
只輸出改寫後標題，每行一個，格式「編號. 標題」，無其他說明。

{numbered}

輸出："""
    else:
        prompt = f"""Below are {len(batch_rows)} clickbait headlines (with body). Rewrite each into a neutral,
objective, non-clickbait news headline.
Requirements: faithfully reflect the body; do not add information absent from the body;
remove hype, curiosity gaps, questions, exclamations, vague pronouns.
Output only the rewritten headlines, one per line, format "N. headline", no other text.

{numbered}

Output:"""
    return prompt


def build_prompt_transition_pair(batch_rows, lang, assigned_words):
    """transition+label=0 → 每則造「一對」：tw_neg(0) + tw_pos(1)，都用指定轉折詞、同內文。
    輸出格式每則兩行：'N-A. 忠實版' 與 'N-B. 標題黨版'。"""
    if lang == "zh":
        numbered = "\n".join(
            f"{i+1}.（轉折詞「{assigned_words[i]}」）內文：{_content_snippet(r)}"
            for i, r in enumerate(batch_rows)
        )
        prompt = f"""以下有 {len(batch_rows)} 篇真實新聞內文。請為每篇產出**兩個**新聞標題，
**兩個標題都必須原封不動地包含該篇指定的轉折詞**（一字不漏），但一個是非標題黨、一個是標題黨：

A（非標題黨, label=0）：用轉折詞做「事實對比」，但資訊完整、無懸念缺口。
   例「台股今日收盤竟上漲50點，終場成交量增兩成」——有「竟」但說完了。
B（標題黨, label=1）：同樣用轉折詞，但刻意遮蔽關鍵資訊、製造懸念（用「這/某/一個」藏住答案）。
   例「台股盤中竟驚見這一幕，股民全看傻」——有「竟」但藏關鍵。

要求：兩者都基於內文真實事實、不得捏造；**兩者都要自然且明確地使用指定轉折詞**。
每篇輸出兩行，格式嚴格為：
{{編號}}-A. 非標題黨標題
{{編號}}-B. 標題黨標題

{numbered}

輸出："""
    else:
        numbered = "\n".join(
            f"{i+1}.(hook \"{assigned_words[i]}\") Body: {_content_snippet(r)}"
            for i, r in enumerate(batch_rows)
        )
        prompt = f"""Below are {len(batch_rows)} real news bodies. For each, produce **two** headlines.
**Both headlines MUST contain the assigned hook phrase verbatim** — one non-clickbait, one clickbait:

A (non-clickbait, label=0): use the hook for factual contrast, but info-complete, no curiosity gap.
B (clickbait, label=1): same hook, but deliberately hides key info / creates suspense
   (use vague references like "this", "one", "a decision" to hide the answer).

Both must be grounded in the body, no fabrication. **Both must clearly use the assigned hook.**
Output two lines per item, strict format:
{{N}}-A. non-clickbait headline
{{N}}-B. clickbait headline

{numbered}

Output:"""
    return prompt


def build_prompt_clickbait(batch_rows, fit_type, lang):
    """label=0 (非 transition) → 造單一 label=1（依 fit_type：academic 知性腔 / normal·none 資訊缺口型）。

    Codex review 2026-06 強化：核心改為「**必須遮蔽關鍵答案**」——明確要求標題讀完
    『無法知道誰/什麼/結果』，逼讀者點擊。否則 Gemini 偷懶產正常摘要。
    """
    numbered = "\n".join(
        f"{i+1}. 標題：{r['title']}\n   內文：{_content_snippet(r)}" for i, r in enumerate(batch_rows)
    )
    if fit_type == "academic":
        if lang == "zh":
            prompt = f"""以下有 {len(batch_rows)} 則真實新聞。請各改寫成帶「知性/探討語氣」的**標題黨**標題。

核心要求（必須做到，否則無效）：**標題必須遮蔽關鍵答案**——用「探討/研究/揭密/解析」開頭，
搭配「某種/某類/這些/這個/背後原因」等模糊代稱，把『到底是什麼/誰/結果如何』藏進內文，
讓人光看標題無法得知答案、必須點進去。
要求：語氣知性但刻意製造資訊缺口；基於內文不得捏造。

例：原「研究：每天走7000步可降低死亡風險」
   → 標題黨「解析：每天一個習慣，背後藏著影響壽命的關鍵」（藏住「走7000步」）
   ✗ 錯誤「探討每天走7000步如何降低死亡風險」（答案還在，沒遮蔽）

只輸出標題，每行一個，格式「編號. 標題」。

{numbered}

輸出："""
        else:
            prompt = f"""Rewrite each of the {len(batch_rows)} items into a **clickbait** headline with an
intellectual/explainer tone ("Exploring / The science behind / What research reveals about / The reason").

Core requirement (mandatory): the headline **must hide the key answer** — use vague references
("a certain", "these", "one reason", "the science behind") so that reading the headline alone
does NOT tell you what/who/the result; the reader must click to find out.
Grounded in the body, no fabrication.

Example: original "Study: walking 7000 steps a day cuts mortality risk"
   → clickbait "The science behind one daily habit that quietly affects your lifespan" (hides "7000 steps")
   ✗ Wrong: "Exploring how walking 7000 steps cuts mortality risk" (answer still present)

Output one headline per line, format "N. headline".

{numbered}

Output:"""
    else:  # normal / none：資訊缺口型（不加誇飾詞）
        if lang == "zh":
            prompt = f"""以下有 {len(batch_rows)} 則真實非標題黨新聞（含內文）。請各改寫成**標題黨**版本。

核心要求（必須做到，否則無效）：**標題必須遮蔽關鍵資訊、製造懸念缺口**。
判定標準：改寫後的標題，讀者光看標題『無法知道關鍵的人/事/物/結果是什麼』，必須點進內文才知道。
若標題把答案說完了（像正常摘要），就是**失敗**，請重做。

禁止：不得加入驚嘆號「！」、誇飾詞（竟、驚見、不可思議、震驚、超強等）、感嘆語氣。
手法：**強制改用模糊代稱（「這件事」「某原因」「一個決定」「這一幕」「這些人」）藏住答案**，
      或用問句/懸念結構讓讀者想點進去，但語氣平靜、外表像正常新聞標題。

例：原「台積電宣布明年擴廠計畫，投資金額達300億」
   → 標題黨「台積電悄悄做了一個決定，業界人士看了沉默」（藏住「擴廠300億」）✓
   ✗ 錯誤1「台積電竟宣布驚人消息！」（加了誇飾詞）
   ✗ 錯誤2「台積電宣布明年擴廠，投資300億」（答案說完了，沒遮蔽 → 失敗）

基於內文真實事實，不得捏造。只輸出標題，每行一個，格式「編號. 標題」。

{numbered}

輸出："""
        else:
            prompt = f"""Rewrite each of the {len(batch_rows)} non-clickbait items (with body) into a **clickbait** version.

Core requirement (mandatory): the headline **must hide the key information / create a curiosity gap**.
Test: after your rewrite, a reader looking at ONLY the headline must NOT be able to tell the key
who/what/result — they must click to find out. If the headline states the answer (like a normal
summary), it is a **FAILURE** — redo it.

Forbidden: NO exclamation marks, NO hype words (shocking, unbelievable, incredible, stunning, etc.),
           NO dramatic adjectives.
Technique: you MUST use vague references ("a decision", "one reason", "what happened", "this move",
           "these people") to hide the key answer, or use a question structure — but keep the tone
           calm, so it reads like a normal headline on the surface.

Example: original "Apple announces $10B investment in new US factories"
   → clickbait "Apple quietly made a decision that caught the industry off guard" (hides the $10B) ✓
   ✗ Wrong 1: "Apple makes SHOCKING announcement!" (uses hype word)
   ✗ Wrong 2: "Apple announces $10B factory investment" (answer stated, no gap → FAILURE)

Grounded in the body, no fabrication. Output one per line, format "N. headline".

{numbered}

Output:"""
    return prompt


def _safe_id(row):
    import hashlib
    base = str(row.get("src_url") or row.get("orig_title") or "")
    h = hashlib.md5(base.encode("utf-8")).hexdigest()[:10]
    return f"{row['lang']}_g8_{h}"


def call_gemini(prompt, expect_n):
    """呼叫 Gemini。回傳原始非空行 list（解析交給呼叫端，因不同 prompt 格式不同）。
    空回應(safety filter)或全失敗 → 回 []。"""
    for attempt in range(1, G8_RETRIES + 1):
        try:
            resp = client.models.generate_content(model=G8_MODEL_ID, contents=prompt)
            text = resp.text
            if not text or not text.strip():
                print("  [SKIP] 空回應（safety filter）")
                return []
            return [l.strip() for l in text.strip().splitlines() if l.strip()]
        except Exception as e:
            print(f"  attempt {attempt}/{G8_RETRIES} failed: {e}")
            if attempt < G8_RETRIES:
                time.sleep(G8_RETRY_SLEEP)
            else:
                return []


def parse_numbered(lines, n):
    """解析 'N. 標題' 格式，回傳長度 n 的 list（缺的為 None）。"""
    out = []
    for i in range(n):
        m = next((l for l in lines if l.startswith(f"{i+1}.")), None)
        out.append(m.split(".", 1)[1].strip().strip('"').strip("'") if m else None)
    return out


def parse_pairs(lines, n):
    """解析 transition 一對格式 'N-A.' / 'N-B.'，回傳 [(neg,pos), ...]（缺的為 None）。"""
    pairs = []
    for i in range(n):
        a = next((l for l in lines if l.startswith(f"{i+1}-A.")), None)
        b = next((l for l in lines if l.startswith(f"{i+1}-B.")), None)
        neg = a.split(".", 1)[1].strip().strip('"').strip("'") if a else None
        pos = b.split(".", 1)[1].strip().strip('"').strip("'") if b else None
        pairs.append((neg, pos))
    return pairs

# tw_neg 專用驗證：只排除模糊代稱，不把轉折詞本身當 gap signal
_TW_NEG_VAGUE_ZH = ["這件事", "這一幕", "某原因", "一個決定", "答案", "真相", "內幕",
                     "背後原因", "點進來", "點擊", "詳情", "繼續看", "做了什麼", "說了什麼"]
_TW_NEG_VAGUE_EN = ["this decision", "one reason", "what happened", "this thing",
                     "something unexpected", "a certain", "click to", "find out"]

def _tw_neg_is_vague(s, lang):
    """回傳 True 表示 tw_neg 仍含模糊代稱（驗證失敗）。"""
    if not s:
        return True
    if lang == "zh":
        return any(p in s for p in _TW_NEG_VAGUE_ZH)
    else:
        low = s.lower()
        return any(p in low for p in _TW_NEG_VAGUE_EN)


In [8]:
# ══════════════════════════════════════════════════════════════════
# 類型 1：農場文語氣但資訊完整 label=0（中文，補 kknews-like non-CB）
# 觸發：lang=='zh' AND fit_type in ['academic','none'] AND gemini=0
# 目的：教模型「農場文語氣 ≠ clickbait，判斷依據是資訊是否完整」
# ══════════════════════════════════════════════════════════════════
def build_prompt_farm_label0(batch_rows):
    numbered = "\n".join(
        f"{i+1}. 原標題：{r['title']}\n   內文：{_content_snippet(r)}" for i, r in enumerate(batch_rows)
    )
    prompt = f"""以下有 {len(batch_rows)} 則真實新聞（含內文）。請將每個標題改寫成「內容農場／公眾號風格」的非標題黨標題。

核心要求：
- 語氣要像內容農場（可以感性、口語、帶情緒，例如「太厲害了」「原來如此」「讓人意外」「很多人不知道」）
- 但標題必須資訊完整——讀者光看標題就能知道關鍵的人/事/結果，不需要點進去
- 不得製造懸念缺口，禁止使用以下藏答案的模糊代稱：「這件事」「這一幕」「某原因」「一個決定」「答案」「真相」「內幕」「全看」
- 基於內文，不得捏造

例：原「研究：每天走7000步可降低死亡風險」
   → 農場風格非標題黨「每天走7000步就能降低死亡風險，很多人都不知道這個習慣有多重要」✓
   ✗ 錯誤「每天做這件事竟然能降低死亡風險，你一定要知道」（用「這件事」藏住答案→變成標題黨）

只輸出改寫後標題，每行一個，格式「編號. 標題」，無其他說明。

{numbered}

輸出："""
    return prompt


# farm 驗證：用窄 forbidden pattern，不反用 cb_has_gap（避免誤殺「這個習慣」等正常指代）
FARM_FORBIDDEN_ZH = [
    "這件事", "這一幕", "某原因", "一個決定", "答案", "真相", "內幕", "全看",
    "背後原因", "你不知道", "點進來", "點擊查看", "詳情見", "繼續看",
]

def farm_has_forbidden(s):
    """回傳 True 表示標題含藏答案短語（= 驗證失敗）。"""
    return any(p in (s or "") for p in FARM_FORBIDDEN_ZH)


# ══════════════════════════════════════════════════════════════════
# 類型 2：英文小報／娛樂 clickbait label=1
# 觸發：lang=='en' AND fit_type in ['normal','none'] AND label=1
# 目的：補英文 tabloid/entertainment clickbait，修 nypost FN
# ══════════════════════════════════════════════════════════════════
def build_prompt_tabloid_clickbait(batch_rows):
    numbered = "\n".join(
        f"{i+1}. Title: {r['title']}\n   Body: {_content_snippet(r)}" for i, r in enumerate(batch_rows)
    )
    prompt = f"""Rewrite each of the {len(batch_rows)} items into an English **web-style clickbait** headline.

Target style:
- webis-clickbait-17 / Business Insider / BuzzFeed / Forbes / viral explainer style
- The headline may sound calm and informational, but it must create a knowledge gap.
- Prefer Who / What / How / Why patterns, not only celebrity gossip.

Use one of these patterns when appropriate:
- "Here's why ..."
- "Here's how ..."
- "This is why ..."
- "What happens when ..."
- "What experts say about ..."
- "A [person/expert/group] explains why ..."
- "How [X] became [Y]"
- "What we know about ..."
- "Who should ..."
- "Why you should ..."
- "Why you shouldn't ..."

Core requirement:
- The headline should reveal the topic, but hide the key explanation, reason, mechanism, ranking, or final answer.
- The reader should know what the article is about, but still need to click to learn the answer.
- Do NOT rely only on vague phrases like "this thing", "one decision", or "what happened next".
- NO exclamation marks, NO ALL CAPS, NO hype words such as shocking, unbelievable, insane, jaw-dropping.

Good examples:
Original: "Cold showers may improve alertness and circulation"
→ "A Navy SEAL explains why you should end a shower with cold water" ✓

Original: "Mistletoe is a parasite that became linked to Christmas through old customs"
→ "Mistletoe is a tree-killing parasite — here's how it became a Christmas tradition" ✓

Original: "A new bank founded in 2010 is challenging the five largest banks"
→ "A bank founded in 2010 is taking on the big five — here's why it may finally turn a profit" ✓

Wrong examples:
✗ "This one thing will change your life" (too generic)
✗ "You won't believe what happened next" (too low-quality)
✗ "Taylor Swift and Travis Kelce's shocking dinner date" (hype + exposed as tabloid only)
✗ "A bank founded in 2010 is taking on the big five and is set to make its first profit" (too info-complete)

Grounded in the body. Do not fabricate facts.
Output one per line, format "N. headline".

{numbered}

Output:"""
    return prompt


# ══════════════════════════════════════════════════════════════════
# 類型 3：英文知性／健康／投資 info-gap clickbait label=1
# 觸發：lang=='en' AND fit_type=='academic' AND label=0
# 目的：補英文知性語氣 info-gap clickbait，修英文 academic FN
# 與既有 build_prompt_clickbait academic 的差異：
#  - 明確限定 health/finance/psychology/workplace 四個 domain
#  - 禁止 hype words，語氣要「calm and credible」（既有版本沒有這個限制）
# ══════════════════════════════════════════════════════════════════
def build_prompt_infogap_clickbait(batch_rows):
    numbered = "\n".join(
        f"{i+1}. Title: {r['title']}\n   Body: {_content_snippet(r)}" for i, r in enumerate(batch_rows)
    )
    prompt = f"""Rewrite each of the {len(batch_rows)} items into an English **intellectual/explainer clickbait** headline
in the style of health, finance, psychology, or workplace content.

Core requirement (mandatory): the headline must hide the key answer behind an intellectual framing —
use phrases like "The science behind", "Why researchers say", "The reason most people",
"What studies reveal about", "The one factor" — but make sure the KEY answer (what/who/how much/result)
is NOT stated; the reader must click to find out.

Forbidden: NO exclamation marks, NO hype words (shocking, unbelievable, stunning),
           NO dramatic adjectives. Keep the tone calm and credible.

Example: original "Study: walking 7000 steps a day cuts mortality risk by 11%"
   → info-gap clickbait "The science behind one daily habit that quietly affects how long you live" ✓
   ✗ Wrong: "Exploring how walking 7000 steps cuts mortality risk" (answer still present)
   ✗ Wrong: "The SHOCKING truth about walking!" (hype word)

Grounded in the body, no fabrication. Output one per line, format "N. headline".

{numbered}

Output:"""
    return prompt


# ══════════════════════════════════════════════════════════════════
# 類型 4：從 transition label=1 clickbait 生 tw_neg（加轉折詞+資訊完整）
# tw_pos 直接用原標題（本身已是 clickbait）
# ══════════════════════════════════════════════════════════════════
def build_prompt_tw_neg_from_label1(batch_rows, lang, assigned_words):
    if lang == "zh":
        numbered = "\n".join(
            f"{i+1}.（轉折詞「{assigned_words[i]}」）標題黨標題：{r['title']}\n   內文：{_content_snippet(r)}"
            for i, r in enumerate(batch_rows)
        )
        prompt = (
            f"以下有 {len(batch_rows)} 個標題黨標題（含內文）。請為每個產出一個**非標題黨**改寫版。\n\n"
            "要求：\n"
            "- **必須原封不動地包含指定轉折詞**（一字不漏）\n"
            "- 資訊完整——讀者光看標題就能知道關鍵的人/事/結果，不需要點進去\n"
            "- 不得製造懸念缺口，不使用模糊代稱（「這件事」「某原因」「一個決定」等）\n"
            "- 語氣自然，基於內文，不得捏造\n\n"
            "例（轉折詞「竟」）：原標題黨「他竟做了這個決定，讓所有人傻眼」\n"
            "   → 非標題黨「他竟在股東大會上宣布辭去執行長一職，即日生效」✓\n"
            "   ✗ 錯誤「他竟做了一個決定」（仍然模糊）\n\n"
            "每篇輸出一行，格式「編號. 標題」。\n\n"
            f"{numbered}\n\n輸出："
        )
    else:
        numbered = "\n".join(
            f"{i+1}.(hook \"{assigned_words[i]}\") Clickbait: {r['title']}\n   Body: {_content_snippet(r)}"
            for i, r in enumerate(batch_rows)
        )
        prompt = (
            f"Below are {len(batch_rows)} clickbait headlines (with body). For each, produce a **non-clickbait** rewrite.\n\n"
            "Requirements:\n"
            "- **Must contain the assigned hook phrase verbatim**\n"
            "- Info-complete: reader knows the key who/what/result from the headline alone\n"
            "- No curiosity gap, no vague references (\"this decision\", \"one reason\", \"what happened\", etc.)\n"
            "- Natural tone, grounded in body, no fabrication\n\n"
            "Example (hook \"turns out\"): clickbait \"Turns out this habit is secretly destroying your health\"\n"
            "   → non-clickbait \"Turns out drinking more than 4 cups of coffee daily raises heart risk by 20%, study finds\" ✓\n"
            "   ✗ Wrong: \"Turns out this one thing affects your health\" (still vague)\n\n"
            "Output one per line, format \"N. headline\".\n\n"
            f"{numbered}\n\nOutput:"
        )
    return prompt


# ══════════════════════════════════════════════════════════════════
# 類型 5：從 none/normal label=0 生 tw_neg only（加轉折詞+資訊完整）
# 目的：大幅增加轉折詞 label=0 訓練樣本，修 Bug 3
# ══════════════════════════════════════════════════════════════════
def build_prompt_tw_neg_only(batch_rows, lang, assigned_words):
    if lang == "zh":
        numbered = "\n".join(
            f"{i+1}.（轉折詞「{assigned_words[i]}」）原標題：{r['title']}\n   內文：{_content_snippet(r)}"
            for i, r in enumerate(batch_rows)
        )
        prompt = (
            f"以下有 {len(batch_rows)} 則非標題黨新聞（含內文）。請將每個標題改寫成包含指定轉折詞的**非標題黨**版本。\n\n"
            "要求：\n"
            "- **必須原封不動地包含指定轉折詞**（一字不漏）\n"
            "- 資訊完整——讀者光看標題就能知道關鍵的人/事/結果\n"
            "- 不得製造懸念缺口，不使用模糊代稱\n"
            "- 語氣自然，基於內文，不得捏造\n\n"
            "例（轉折詞「居然」）：原「台積電宣布擴廠，投資300億」\n"
            "   → 「台積電居然宣布斥資300億擴廠，預計2026年完工」✓\n"
            "   ✗ 錯誤「台積電居然做了這個決定」（仍然模糊）\n\n"
            "每篇輸出一行，格式「編號. 標題」。\n\n"
            f"{numbered}\n\n輸出："
        )
    else:
        numbered = "\n".join(
            f"{i+1}.(hook \"{assigned_words[i]}\") Original: {r['title']}\n   Body: {_content_snippet(r)}"
            for i, r in enumerate(batch_rows)
        )
        prompt = (
            f"Below are {len(batch_rows)} non-clickbait headlines (with body). Rewrite each to include the assigned hook phrase while keeping it non-clickbait.\n\n"
            "Requirements:\n"
            "- **Must contain the assigned hook phrase verbatim**\n"
            "- Info-complete: reader knows the key who/what/result from the headline alone\n"
            "- No curiosity gap, no vague references\n"
            "- Natural tone, grounded in body, no fabrication\n\n"
            "Example (hook \"surprisingly\"): original \"Apple reports record quarterly revenue\"\n"
            "   → \"Apple surprisingly reported record revenue of $124B, beating analyst estimates by 8%\" ✓\n"
            "   ✗ Wrong: \"Apple surprisingly did something unexpected\" (still vague)\n\n"
            "Output one per line, format \"N. headline\".\n\n"
            f"{numbered}\n\nOutput:"
        )
    return prompt

## Step 3：主迴圈（checkpoint 斷點續跑）

每筆結果逐筆寫進 `_adv_g8_checkpoint.jsonl` 並 flush；重跑時讀回 `done_ids` 跳過已完成。
分組鍵 = `(fit_type, lang)`，同組同 batch 共用一個 prompt。

In [9]:
# 載入 checkpoint（斷點續跑）。⚠️ 邏輯改版後請手動刪舊 _adv_g8_checkpoint.jsonl 再跑
done_ids = set()
g8_rows = []
if G8_CHECKPOINT.exists():
    with open(G8_CHECKPOINT, encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            cid = row["ckpt_id"]
            if cid not in done_ids:          # 載入時也去重，防 cell 重複執行累積
                done_ids.add(cid)
                g8_rows.append(row)
    print(f"從 checkpoint 續跑：已完成 {len(g8_rows)} 筆")

records = base.to_dict("records")


def emit(ckpt, **kw):
    if kw["ckpt_id"] in done_ids:           # 防止同一 session 重複執行 cell 時重複寫入
        return
    done_ids.add(kw["ckpt_id"])
    g8_rows.append(kw)
    ckpt.write(json.dumps(kw, ensure_ascii=False) + "\n")


def batched(seq, n):
    for i in range(0, len(seq), n):
        yield seq[i:i + n]


reject_stats = {"cb_retry": 0, "cb_drop": 0, "pair_retry": 0, "pair_drop": 0,
                "farm_retry": 0, "farm_drop": 0,
                "infogap_retry": 0, "infogap_drop": 0,
                "tabloid_retry": 0, "tabloid_drop": 0}


def synth_cb_batch(batch, fit_type, lang):
    cbs = parse_numbered(call_gemini(build_prompt_clickbait(batch, fit_type, lang), len(batch)), len(batch))
    bad = [i for i, (r, cb) in enumerate(zip(batch, cbs))
           if not cb_has_gap(cb, r["title"], lang)[0]]
    if bad:
        reject_stats["cb_retry"] += len(bad)
        retry_rows = [batch[i] for i in bad]
        retry_cbs = parse_numbered(
            call_gemini(build_prompt_clickbait(retry_rows, fit_type, lang), len(retry_rows)),
            len(retry_rows))
        for j, i in enumerate(bad):
            rc = retry_cbs[j]
            if cb_has_gap(rc, batch[i]["title"], lang)[0]:
                cbs[i] = rc
            else:
                cbs[i] = None
                reject_stats["cb_drop"] += 1
    return cbs


def synth_pair_batch(batch, lang, words):
    pairs = parse_pairs(call_gemini(build_prompt_transition_pair(batch, lang, words), len(batch)), len(batch))
    bad = [i for i, (neg, pos) in enumerate(pairs)
           if not pair_ok(neg, pos, words[i], lang)[0]]
    if bad:
        reject_stats["pair_retry"] += len(bad)
        retry_rows = [batch[i] for i in bad]
        retry_words = [words[i] for i in bad]
        retry_pairs = parse_pairs(
            call_gemini(build_prompt_transition_pair(retry_rows, lang, retry_words), len(retry_rows)),
            len(retry_rows))
        for j, i in enumerate(bad):
            rn, rp = retry_pairs[j]
            if pair_ok(rn, rp, words[i], lang)[0]:
                pairs[i] = (rn, rp)
            else:
                pairs[i] = (None, None)
                reject_stats["pair_drop"] += 1
    return pairs


def synth_farm_batch(batch):
    farms = parse_numbered(call_gemini(build_prompt_farm_label0(batch), len(batch)), len(batch))
    bad = [i for i, f in enumerate(farms) if not f or farm_has_forbidden(f)]
    if bad:
        reject_stats["farm_retry"] += len(bad)
        retry_rows = [batch[i] for i in bad]
        retry_farms = parse_numbered(call_gemini(build_prompt_farm_label0(retry_rows), len(retry_rows)), len(retry_rows))
        for j, i in enumerate(bad):
            rf = retry_farms[j]
            if rf and not farm_has_forbidden(rf):
                farms[i] = rf
            else:
                farms[i] = None
                reject_stats["farm_drop"] += 1
    return farms


def synth_infogap_batch(batch):
    cbs = parse_numbered(call_gemini(build_prompt_infogap_clickbait(batch), len(batch)), len(batch))
    bad = [i for i, (r, cb) in enumerate(zip(batch, cbs))
           if not cb_has_gap(cb, r["title"], "en")[0]]
    if bad:
        reject_stats["infogap_retry"] += len(bad)
        retry_rows = [batch[i] for i in bad]
        retry_cbs = parse_numbered(call_gemini(build_prompt_infogap_clickbait(retry_rows), len(retry_rows)), len(retry_rows))
        for j, i in enumerate(bad):
            rc = retry_cbs[j]
            if cb_has_gap(rc, batch[i]["title"], "en")[0]:
                cbs[i] = rc
            else:
                cbs[i] = None
                reject_stats["infogap_drop"] += 1
    return cbs


def synth_tabloid_batch(batch):
    tabs = parse_numbered(call_gemini(build_prompt_tabloid_clickbait(batch), len(batch)), len(batch))
    bad = [i for i, (r, t) in enumerate(zip(batch, tabs))
           if not cb_has_gap(t, r["title"], "en")[0]]
    if bad:
        reject_stats["tabloid_retry"] += len(bad)
        retry_rows = [batch[i] for i in bad]
        retry_tabs = parse_numbered(call_gemini(build_prompt_tabloid_clickbait(retry_rows), len(retry_rows)), len(retry_rows))
        for j, i in enumerate(bad):
            rt = retry_tabs[j]
            if cb_has_gap(rt, batch[i]["title"], "en")[0]:
                tabs[i] = rt
            else:
                tabs[i] = None
                reject_stats["tabloid_drop"] += 1
    return tabs


TABLOID_CAP = 15

with open(G8_CHECKPOINT, "a", encoding="utf-8") as ckpt:

    gemini0 = [r for r in records if int(r.get("gemini", 1)) == 0]
    for r in gemini0:
        ckpt_id = f"{r.get('url', r['title'])}_direct"
        if ckpt_id in done_ids:
            continue
        emit(ckpt, ckpt_id=ckpt_id,
             orig_title=r["title"], orig_label=int(r["label"]), content=r.get("content", ""),
             lang=r["lang"], fit_type=r["fit_type"], kind="direct",
             real_label=int(r["label"]), real_tag="real",
             synth_title=None, synth_label=None, synth_tag=None,
             assigned_word="")
    print(f"[direct] gemini=0 直接 emit：{len(gemini0)} 筆")
    ckpt.flush()

    gemini1_records = [r for r in records if int(r.get("gemini", 1)) == 1]

    for fit_type in ["transition", "academic", "normal", "none"]:
        for lang in ["zh", "en"]:
            group = [r for r in gemini1_records if r["fit_type"] == fit_type and r["lang"] == lang]
            if not group:
                continue
            label0 = [r for r in group if int(r["label"]) == 0]
            label1 = [r for r in group if int(r["label"]) == 1]

            pend1 = [r for r in label1 
                if f"{r.get('url', r['title'])}_tone" not in done_ids
                and f"{r.get('url', r['title'])}_twpair" not in done_ids]
            if pend1:
                if fit_type == "transition":
                    # tw_pos = orig title (already clickbait), Gemini generates tw_neg only
                    print(f"[{fit_type}/{lang}] tw_neg_from_label1 {len(pend1)}/{len(label1)}")
                    words = SURPRISE_WORDS_ZH if lang == "zh" else SURPRISE_WORDS_EN
                    assigned = [words[k % len(words)] for k in range(len(pend1))]
                    for bi, batch in enumerate(batched(pend1, G8_BATCH_SIZE)):
                        bw = assigned[bi * G8_BATCH_SIZE: bi * G8_BATCH_SIZE + len(batch)]
                        negs = parse_numbered(call_gemini(build_prompt_tw_neg_from_label1(batch, lang, bw), len(batch)), len(batch))
                        for r, neg, w in zip(batch, negs, bw):
                            wc = w if lang == "zh" else w.lower()
                            neg_ok = (neg and wc in (neg if lang == "zh" else neg.lower())
                                      and not _tw_neg_is_vague(neg, lang))
                            emit(ckpt, ckpt_id=f"{r.get('url', r['title'])}_twpair",
                                 orig_title=r["title"], orig_label=1, content=r.get("content", ""),
                                 lang=lang, fit_type=fit_type, kind="tw_pair",
                                 tw_neg=neg if neg_ok else None, tw_pos=r["title"],
                                 assigned_word=w)
                        ckpt.flush(); time.sleep(G8_SLEEP)
                else:
                    print(f"[{fit_type}/{lang}] tone(label=1→plain) {len(pend1)}/{len(label1)}")
                    for batch in batched(pend1, G8_BATCH_SIZE):
                        plains = parse_numbered(call_gemini(build_prompt_plain(batch, lang), len(batch)), len(batch))
                        for r, plain in zip(batch, plains):
                            emit(ckpt, ckpt_id=f"{r.get('url', r['title'])}_tone",
                                 orig_title=r["title"], orig_label=1, content=r.get("content", ""),
                                 lang=lang, fit_type=fit_type, kind="tone",
                                 real_label=1, real_tag="tone_orig",
                                 synth_title=plain, synth_label=0, synth_tag="tone_plain",
                                 assigned_word="")
                        ckpt.flush(); time.sleep(G8_SLEEP)

            pend0 = [r for r in label0 if f"{r.get('url', r['title'])}_cb" not in done_ids]
            if not pend0:
                continue

            if fit_type == "transition":
                print(f"[{fit_type}/{lang}] pair(tw_neg+tw_pos) {len(pend0)}/{len(label0)}")
                words = SURPRISE_WORDS_ZH if lang == "zh" else SURPRISE_WORDS_EN
                assigned = [words[k % len(words)] for k in range(len(pend0))]
                for bi, batch in enumerate(batched(pend0, G8_BATCH_SIZE)):
                    bw = assigned[bi * G8_BATCH_SIZE: bi * G8_BATCH_SIZE + len(batch)]
                    pairs = synth_pair_batch(batch, lang, bw)
                    for r, (neg, pos), w in zip(batch, pairs, bw):
                        emit(ckpt, ckpt_id=f"{r.get('url', r['title'])}_cb",
                             orig_title=r["title"], orig_label=0, content=r.get("content", ""),
                             lang=lang, fit_type=fit_type, kind="tw_pair",
                             tw_neg=neg, tw_pos=pos, assigned_word=w)
                    ckpt.flush(); time.sleep(G8_SLEEP)

            elif fit_type == "academic" and lang == "en":
                print(f"[{fit_type}/{lang}] infogap_clickbait(label=0→1) {len(pend0)}/{len(label0)}")
                for batch in batched(pend0, G8_BATCH_SIZE):
                    cbs = synth_infogap_batch(batch)
                    for r, cb in zip(batch, cbs):
                        emit(ckpt, ckpt_id=f"{r.get('url', r['title'])}_cb",
                             orig_title=r["title"], orig_label=0, content=r.get("content", ""),
                             lang=lang, fit_type=fit_type, kind="cb",
                             real_label=0, real_tag="real",
                             synth_title=cb, synth_label=1, synth_tag="infogap",
                             assigned_word="")
                    ckpt.flush(); time.sleep(G8_SLEEP)

            elif fit_type in ["normal", "none"] and lang == "en":
                if pend0:
                    print(f"[{fit_type}/{lang}] clickbait(label=0→1) {len(pend0)}")
                    for batch in batched(pend0, G8_BATCH_SIZE):
                        cbs = synth_cb_batch(batch, fit_type, lang)
                        for r, cb in zip(batch, cbs):
                            emit(ckpt, ckpt_id=f"{r.get('url', r['title'])}_cb",
                                 orig_title=r["title"], orig_label=0, content=r.get("content", ""),
                                 lang=lang, fit_type=fit_type, kind="cb",
                                 real_label=0, real_tag="real",
                                 synth_title=cb, synth_label=1, synth_tag="norm",
                                 assigned_word="")
                        ckpt.flush(); time.sleep(G8_SLEEP)

                pend1_tabloid = [r for r in label1
                                 if f"{r.get('url', r['title'])}_tabloid" not in done_ids][:TABLOID_CAP]
                if pend1_tabloid:
                    print(f"[{fit_type}/{lang}] tabloid_clickbait(cap={TABLOID_CAP}) {len(pend1_tabloid)}")
                    for batch in batched(pend1_tabloid, G8_BATCH_SIZE):
                        tabs = synth_tabloid_batch(batch)
                        for r, tab in zip(batch, tabs):
                            emit(ckpt, ckpt_id=f"{r.get('url', r['title'])}_tabloid",
                                 orig_title=r["title"], orig_label=1, content=r.get("content", ""),
                                 lang=lang, fit_type=fit_type, kind="cb",
                                 real_label=1, real_tag="real",
                                 synth_title=tab, synth_label=1, synth_tag="tabloid",
                                 assigned_word="")
                        ckpt.flush(); time.sleep(G8_SLEEP)

            else:
                print(f"[{fit_type}/{lang}] clickbait(label=0→1) {len(pend0)}/{len(label0)}")
                for batch in batched(pend0, G8_BATCH_SIZE):
                    cbs = synth_cb_batch(batch, fit_type, lang)
                    tag = "acad" if fit_type == "academic" else "norm"
                    for r, cb in zip(batch, cbs):
                        emit(ckpt, ckpt_id=f"{r.get('url', r['title'])}_cb",
                             orig_title=r["title"], orig_label=0, content=r.get("content", ""),
                             lang=lang, fit_type=fit_type, kind="cb",
                             real_label=0, real_tag="real",
                             synth_title=cb, synth_label=1, synth_tag=tag,
                             assigned_word="")
                    ckpt.flush(); time.sleep(G8_SLEEP)

    # tw_neg_only: none/normal label=0, add transition word, label=0 only
    tw_neg_cands = [r for r in records
                    if int(r.get("gemini", 1)) == 1 and int(r["label"]) == 0
                    and r["fit_type"] in ["none", "normal"]]
    pend_twneg = [r for r in tw_neg_cands if f"{r.get('url', r['title'])}_twneg" not in done_ids]
    if pend_twneg:
        print(f"[tw_neg_only] none+normal label=0: {len(pend_twneg)}")
        for _lang in ["zh", "en"]:
            lang_rows = [r for r in pend_twneg if r["lang"] == _lang]
            if not lang_rows: continue
            words = SURPRISE_WORDS_ZH if _lang == "zh" else SURPRISE_WORDS_EN
            assigned = [words[k % len(words)] for k in range(len(lang_rows))]
            for bi, batch in enumerate(batched(lang_rows, G8_BATCH_SIZE)):
                bw = assigned[bi * G8_BATCH_SIZE: bi * G8_BATCH_SIZE + len(batch)]
                negs = parse_numbered(call_gemini(build_prompt_tw_neg_only(batch, _lang, bw), len(batch)), len(batch))
                for r, neg, w in zip(batch, negs, bw):
                    wc = w if _lang == "zh" else w.lower()
                    if neg and wc in (neg if _lang == "zh" else neg.lower()) and not _tw_neg_is_vague(neg, _lang):
                        emit(ckpt, ckpt_id=f"{r.get('url', r['title'])}_twneg",
                             orig_title=r["title"], orig_label=0, content=r.get("content", ""),
                             lang=_lang, fit_type=r["fit_type"], kind="tw_neg_only",
                             tw_neg=neg, tw_pos=None, assigned_word=w)
                ckpt.flush(); time.sleep(G8_SLEEP)

        farm_candidates = [r for r in records
                       if int(r.get("gemini", 1)) == 0
                       and r["lang"] == "zh"
                       and int(r["label"]) == 0
                       and r["fit_type"] in ["academic", "none"]]
    pend_farm = [r for r in farm_candidates if f"{r.get('url', r['title'])}_farm" not in done_ids]
    if pend_farm:
        print(f"[類型1/farm/zh] 農場文 label=0 合成 {len(pend_farm)} 筆")
        for batch in batched(pend_farm, G8_BATCH_SIZE):
            farms = synth_farm_batch(batch)
            for r, farm in zip(batch, farms):
                if farm:
                    emit(ckpt, ckpt_id=f"{r.get('url', r['title'])}_farm",
                         orig_title=r["title"], orig_label=0, content=r.get("content", ""),
                         lang="zh", fit_type=r["fit_type"], kind="farm",
                         synth_title=farm, synth_label=0, synth_tag="farm",
                         assigned_word="")
        ckpt.flush(); time.sleep(G8_SLEEP)

g8_df = pd.DataFrame(g8_rows)
print(f"\n處理完成：{len(g8_df)} 筆來源")
print(f"  驗證統計（重試/丟棄）：{reject_stats}")
if len(g8_df):
    print("  kind 分布:", dict(g8_df["kind"].value_counts()))
    tw = g8_df[g8_df["kind"] == "tw_pair"]
    if len(tw):
        print("  transition 轉折詞分配:", dict(tw["assigned_word"].value_counts()))

從 checkpoint 續跑：已完成 706 筆
[direct] gemini=0 直接 emit：42 筆
[transition/zh] pair(tw_neg+tw_pos) 2/20
[tw_neg_only] none+normal label=0: 18


KeyboardInterrupt: 

## Step 4：人工抽查（Human Gate）

In [ ]:
# Human Gate：輸出全部合成品供人工逐筆複核（規則驗證只做粗篩）
# 每列附 gap_check 欄＝程式驗證結果（ok / no_gap_signal / neg_missing_hook…），方便你快速掃。
rows = []
for _, r in g8_df.iterrows():
    lang = r["lang"]
    if r["kind"] == "tw_pair":
        neg, pos, w = r.get("tw_neg"), r.get("tw_pos"), r.get("assigned_word", "")
        chk = pair_ok(neg, pos, w, lang)[1]
        rows.append({"kind": "tw_pair", "fit_type": r["fit_type"], "word": w,
                     "orig": "", "synth": "", "orig_L": "", "synth_L": "",
                     "neg(L0)": neg, "pos(L1)": pos, "gap_check": chk,
                     "content": str(r["content"])[:120]})
    else:
        synth = r.get("synth_title")
        # cb 才需 gap 檢查；tone 是平實化（label=0），不適用 gap，標 n/a
        chk = cb_has_gap(synth, r.get("orig_title", ""), lang)[1] if r["kind"] == "cb" else "n/a(tone)"
        rows.append({"kind": r["kind"], "fit_type": r["fit_type"], "word": "",
                     "orig": r["orig_title"], "synth": synth,
                     "orig_L": r.get("real_label"), "synth_L": r.get("synth_label"),
                     "neg(L0)": "", "pos(L1)": "", "gap_check": chk,
                     "content": str(r["content"])[:120]})
check = pd.DataFrame(rows)

# ── 驗證器良率（review）：cb synth=None、tw_pair 任一端 None ──
cb_all = g8_df[g8_df["kind"] == "cb"]
cb_dropped = sum(1 for _, r in cb_all.iterrows() if not (r.get("synth_title") and str(r.get("synth_title")).strip()))
tw_all = g8_df[g8_df["kind"] == "tw_pair"]
tw_dropped = sum(1 for _, r in tw_all.iterrows()
                 if not (r.get("tw_neg") and str(r.get("tw_neg")).strip()
                         and r.get("tw_pos") and str(r.get("tw_pos")).strip()))
print("── 驗證器良率 ──")
print(f"  cb     : {len(cb_all)-cb_dropped}/{len(cb_all)} 合格（丟棄 {cb_dropped}）")
print(f"  tw_pair: {len(tw_all)-tw_dropped}/{len(tw_all)} 配對完整（丟棄 {tw_dropped}）")

# 全量輸出（不抽樣）。依 kind+fit_type 排序，方便你分組逐筆看。
if len(check):
    check = check.sort_values(["kind", "fit_type", "gap_check"]).reset_index(drop=True)
    out = Path("../dataset/processed/_adv_g8_check.csv")
    check.to_csv(out, index=False, encoding="utf-8-sig")
    print(f"\n全量抽查檔已輸出 {len(check)} 筆 → {out.name}（無抽樣，請逐筆複核）")
    print("欄位 gap_check：ok=過規則篩；其他=被丟棄原因（這些 synth 不會進最終資料）")
    print("\n人工複核重點：")
    print("  (1) 極性正確  (2) 未捏造內文沒有的資訊")
    print("  (3) tw_pair：neg/pos 都帶轉折詞、neg 資訊完整 / pos 有懸念")
    print("  (4) tone：synth 真的平實化（去誇飾懸念）")
    print("  (5) cb：synth 確實遮蔽關鍵答案（非正常摘要）")
    print("\n※ 若發現某筆規則判 ok 但你認為不合格 → 直接在 CSV 標記，組裝前手動排除")
else:
    print("無可抽查樣本")

## Human Gate

請開啟 `dataset/processed/_adv_g8_check.csv` 確認改寫品質後，再執行下一個 cell 組裝輸出。

## Step 5：組裝對抗資料並輸出

每筆來源產生：
1. **真標題**（保留，依 fit_type 決定角色：transition→tw_neg、其餘→原 label）
2. **合成反極標題**（Gemini 產，label=合成目標）

兩端共享同一篇真實內文。輸出 `dataset/processed/adversarial_g8.csv`，schema 對齊主資料集。

In [11]:
# 依 kind 組裝最終對抗資料。五種 kind：
#   tone    : 真標題 tone_orig(1) + 合成 tone_plain(0)
#   cb      : 真標題 real(0/1)   + 合成 clickbait(1)（含 infogap/tabloid/norm/acad）
#   tw_pair : 合成 tw_neg(0) + 合成 tw_pos(1)
#   direct  : gemini=0，只保留真標題（原始 label），不產合成對
#   farm    : 只輸出合成 farm 標題(label=0)，真標題已在 direct 輸出，不重複
direct_sids = {
    _safe_id(row)
    for _, row in g8_df.iterrows()
    if row["kind"] == "direct"
}
records_out = []
for _, row in g8_df.iterrows():
    kind = row["kind"]
    sid = _safe_id(row)
    content, lang = row["content"], row["lang"]

    if kind == "direct":
        orig = str(row.get("orig_title", "")).strip()
        if orig:
            records_out.append(dict(id=f"{sid}_direct", lang=lang, title=orig,
                                    content=content, label=int(row["real_label"]), source="adversarial_g8"))

    elif kind == "farm":
        # 只輸出合成 farm 標題，不輸出 real side（避免與 direct 重複）
        syn = row.get("synth_title")
        if syn and str(syn).strip():
            records_out.append(dict(id=f"{sid}_farm", lang=lang, title=str(syn).strip(),
                                    content=content, label=0, source="adversarial_g8"))

    elif kind == "tw_neg_only":
        neg = row.get("tw_neg")
        if neg and str(neg).strip():
            records_out.append(dict(id=f"{sid}_twneg", lang=lang, title=str(neg).strip(),
                                    content=content, label=0, source="adversarial_g8"))

    elif kind == "tw_pair":
        neg, pos = row.get("tw_neg"), row.get("tw_pos")
        has_neg = neg and str(neg).strip()
        has_pos = pos and str(pos).strip()
        if has_neg:
            records_out.append(dict(id=f"{sid}_tw_neg", lang=lang, title=neg, content=content, label=0, source="adversarial_g8"))
        if has_pos:
            records_out.append(dict(id=f"{sid}_tw_pos", lang=lang, title=pos, content=content, label=1, source="adversarial_g8"))
        if not has_neg and not has_pos:
            orig = str(row.get("orig_title", "")).strip()
            if orig:
                records_out.append(dict(id=f"{sid}_real", lang=lang, title=orig, content=content, label=0, source="adversarial_g8"))

    else:
        # tone / cb：direct 已保留同來源真標題時，跳過 real side，避免髒 checkpoint 產生重複真標題
        if sid not in direct_sids:
            records_out.append(dict(id=f"{sid}_{row['real_tag']}", lang=lang, title=row["orig_title"],
                                    content=content, label=int(row["real_label"]), source="adversarial_g8"))
        syn = row.get("synth_title")
        if syn and str(syn).strip():
            records_out.append(dict(id=f"{sid}_{row['synth_tag']}", lang=lang, title=syn,
                                    content=content, label=int(row["synth_label"]), source="adversarial_g8"))

adv_g8 = pd.DataFrame(records_out)

# 在 drop_duplicates 前 assert/report duplicate ids，不靠 drop 掩蓋 collision
dup_ids = adv_g8[adv_g8["id"].duplicated(keep=False)]
if len(dup_ids):
    print(f"⚠️  發現 {len(dup_ids)} 筆重複 id，請檢查：")
    print(dup_ids[["id", "label", "title"]].to_string())
adv_g8 = adv_g8.drop_duplicates(subset=["id"]).reset_index(drop=True)

assert set(adv_g8["label"].unique()) <= {0, 1}, "label 只能 0/1"
assert list(adv_g8.columns) == ["id", "lang", "title", "content", "label", "source"]
assert adv_g8["lang"].isin(["zh", "en"]).all(), "lang 只能 zh/en"
assert adv_g8["id"].is_unique, "id 必須唯一"

out_path = Path("../dataset/processed/adversarial_g8.csv")
adv_g8.to_csv(out_path, index=False)
print(f"G8 對抗資料輸出：{out_path}（{len(adv_g8)} 筆）")
print(f"  label 分布：\n{adv_g8['label'].value_counts()}")
print(f"  lang 分布：\n{adv_g8['lang'].value_counts()}")
import collections
suf = collections.Counter(i.rsplit("_", 1)[-1] for i in adv_g8["id"])
print(f"  類型分布：{dict(suf)}")

G8 對抗資料輸出：..\dataset\processed\adversarial_g8.csv（1110 筆）
  label 分布：
label
0    636
1    474
Name: count, dtype: int64
  lang 分布：
lang
zh    670
en    440
Name: count, dtype: int64
  類型分布：{'direct': 42, 'orig': 113, 'plain': 113, 'neg': 50, 'pos': 70, 'real': 268, 'acad': 24, 'fixed': 31, 'infogap': 12, 'norm': 197, 'tabloid': 13, 'farm': 20, 'twneg': 157}
